Comparaison SGD / RMSProp / Adam sur un MLP (MNIST)
 
Expérience 1 : courbes de perte, précision test et temps d'entraînement
               avec un learning rate "raisonnable" par optimiseur.
Expérience 2 : sensibilité au learning rate (grille de lr par optimiseur).


In [8]:
# Importations
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt

**I) Déclaration des paramètres**

In [ ]:
BATCH_SIZE = 128 # Groupes de taille 128
METHODES = ["SGD", "RMSProp", "Adam"]
 
GRILLE_LR = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]
EPOCHS_LR = 5          # époques pour l'étude du learning rate
N_SEEDS_LR = 2         # répétitions pour l'étude du learning rate
EPOCHS = 10            # époques pour la comparaison finale / nbr de passage complet syr le jeu de donnée
N_SEEDS = 3            # répétitions (moyenne +/- écart-type)
EPOCHS_VERIF = 5       # époques pour la vérification numpy vs torch.optim
 
# hyperparamètres fixes (valeurs usuelles)
BETA_RMS = 0.9                     # RMSProp, "historiquement" à 0.9
BETA1, BETA2 = 0.9, 0.999          # Adam, les BETA ∈ [0,1)
EPS = 1e-8


**II) Règle de MAJ**  
  
Dans l'esprit graddesc() du TP1

In [ ]:
def regle_sgd(w, g, etat, lr):
    return w - lr * g

# Pour SGD : w_{k+1} = w_k - lr * g, pas (lr) fixe, g est la fonction de perte
# l ou g

 
def regle_rmsprop(w, g, etat, lr, BETA=BETA_RMS, epsilon=EPS):
    v = etat.get("v", np.zeros_like(w)) # récupère v dans le dictionnaire, indispensable car la formule a besoin de la valeur précédente de v
    v = BETA * v + (1 - BETA) * g ** 2
    etat["v"] = v # Range le nv V dans le dictionnaire
    return w - lr * g / (np.sqrt(v) + epsilon)

# Pour RMSProp : v_n+1 <- BETA*v_n + (1-BETA)*g^2 ;  teta_n+1 <- teta_n - lr * g / (sqrt(v_n+1) + eps)
 
 
def regle_adam(w, g, etat, lr, beta1=BETA1, beta2=BETA2, epsilon=EPS):
    t = etat.get("t", 0) + 1 # Création dictionnaire
    m = beta1 * etat.get("m", np.zeros_like(w)) + (1 - beta1) * g # Création dictionnaire
    v = beta2 * etat.get("v", np.zeros_like(w)) + (1 - beta2) * g ** 2 # Création dictionnaire
    etat.update(t=t, m=m, v=v)
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    return w - lr * m_hat / (np.sqrt(v_hat) + epsilon)

# Adam : moments m (ordre 1) et v (ordre 2) + correction de biais (t = numéro d'itération)
# m_n+1 <- BETA1*m_n + (1-BETA1)*g^2 ;  
# v_n+1 <- BETA2*v_n + (1-BETA2)*g^2 ;  
# teta_n+1 <- teta_n - lr * m_hat / (sqrt(v_hat) + eps)

 
 
REGLES = {"SGD": regle_sgd, "RMSProp": regle_rmsprop, "Adam": regle_adam}
 
 
class OptimiseurNumpy:
    """
    Même interface que torch.optim (zero_grad / step), donc utilisable dans SimpleTrainer.
    step() : gradient (autograd) -> vecteur numpy -> règle de mise à jour -> poids du réseau.
    """
    def __init__(self, params, nom, lr):
        self.params = list(params)
        self.regle = REGLES[nom]
        self.lr = lr
        self.etat = {}
        self._depl, self._n = 0.0, 0
 
    def zero_grad(self):
        for p in self.params:
            p.grad = None
 
    @torch.no_grad()
    def step(self):
        w = torch.cat([p.reshape(-1) for p in self.params]).numpy()
        g = torch.cat([p.grad.reshape(-1) for p in self.params]).numpy()
        w_next = self.regle(w, g, self.etat, self.lr)
        # critère de déplacement du TP : ||w_{k+1} - w_k||
        self._depl += float(np.sqrt(np.sum((w_next - w) ** 2)))
        self._n += 1
        i = 0
        for p in self.params:
            k = p.numel()
            p.copy_(torch.from_numpy(w_next[i:i + k]).view_as(p))
            i += k
 
    def deplacement_moyen(self):
        d = self._depl / max(self._n, 1)
        self._depl, self._n = 0.0, 0
        return d
 
 
def make_optimizer(nom, model, lr, backend="numpy"):
    if backend == "numpy":
        return OptimiseurNumpy(model.parameters(), nom, lr)
    # version PyTorch (sert uniquement à vérifier nos règles)
    if nom == "SGD":
        return torch.optim.SGD(model.parameters(), lr=lr)
    if nom == "RMSProp":
        return torch.optim.RMSprop(model.parameters(), lr=lr, alpha=RHO, eps=EPS)
    if nom == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr, betas=(BETA1, BETA2), eps=EPS)
    raise ValueError(nom)


**II) Chargement des données + modèle**
    
MNIST est un jeu de donnée qui contient 70 000 images en niveaux de gris qui répresentent des chiffres manuscrits de 0 à 9.  
Il y a 10 classes au total.  
Il y a 60 000 images pour les données d'entraînement et 10 000 images pour les données test.  
C'est une base de donnée très utilisée en apprentissage automatique, pour tester des algorithmes de classification.  

In [ ]:
from torchvision import datasets

def load_mnist():
    def prep(ds):
        X = ds.data.float().div(255.0)
        X = (X - 0.1307) / 0.3081
        return X.view(len(X), -1), ds.targets # équivalent au transforms.Lambda(lambda x: x.view(-1)) du TD de deep
    train = datasets.MNIST(root="./data", train=True, download=True)
    test = datasets.MNIST(root="./data", train=False, download=True)
    return prep(train), prep(test)
 
 
def make_loader(data, shuffle, seed=0, batch_size=BATCH_SIZE):
    X, y = data
    g = torch.Generator().manual_seed(seed)       # même ordre des batchs pour toutes les méthodes
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle, generator=g)

 
 

# Modèle MLP

class MulticlassMLP(nn.Module):
    def __init__(self, n_features=784, hidden_1=128, hidden_2=64, n_classes=10):
        super().__init__()
        self.cachee1 = nn.Linear(n_features, hidden_1)
        self.cachee2 = nn.Linear(hidden_1, hidden_2)
        self.act = nn.ReLU()
        self.sortie = nn.Linear(hidden_2, n_classes)
 
    def forward(self, x):
        h = self.act(self.cachee1(x))
        h = self.act(self.cachee2(h))
        return self.sortie(h)
    
# n_classes : nbr de classes en sorties
# hidden_1 : couche cachee 1
# hidden_2 : couche cachee 2

 
 
class SimpleTrainer:
    def __init__(self, model, optimizer, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.history = {"train_loss": [], "val_loss": [], "val_acc": [],
                        "time": [], "deplacement": []}
 
    def _run_epoch(self, loader, train):
        self.model.train(train)
        total_loss, total_ok, n = 0.0, 0, 0
        with torch.set_grad_enabled(train):
            for xb, yb in loader:
                preds = self.model(xb)
                loss = self.loss_fn(preds, yb)
                if train:      # en mode entraînement : rétropropagation + mise à jour des poids
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()
                total_loss += loss.item() * xb.size(0)
                total_ok += (preds.argmax(1) == yb).sum().item()
                n += xb.size(0)
        return total_loss / n, total_ok / n # Renvoie la perte et la précision
 
    def fit(self, train_loader, val_loader, epochs):
        t0 = time.perf_counter()
        for _ in range(epochs):
            tr_loss, _ = self._run_epoch(train_loader, train=True)
            va_loss, va_acc = self._run_epoch(val_loader, train=False) # évaluation sur validation / test
            h = self.history
            h["train_loss"].append(tr_loss)
            h["val_loss"].append(va_loss)
            h["val_acc"].append(va_acc)
            h["time"].append(time.perf_counter() - t0)          # temps cumulé
            h["deplacement"].append(getattr(self.optimizer, "deplacement_moyen", lambda: np.nan)())
        return self.history
 
# entraînement complet
# Même seed => mêmes poids initiaux et mêmes mini-batchs
def run(nom, lr, train_data, val_data, epochs, seed, backend="numpy"):
    torch.manual_seed(seed)
    model = MulticlassMLP(n_features=train_data[0].shape[1])
    opt = make_optimizer(nom, model, lr, backend)
    trainer = SimpleTrainer(model, opt, nn.CrossEntropyLoss())
    return trainer.fit(make_loader(train_data, True, seed), make_loader(val_data, False, 0, 1000), epochs)





**Expérience 1 : sensibilité au learning rate**

Rappel : GRILLE_LR = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]

In [ ]:
def experiment_lr_sensitivity(train_data):
    # validation = les 10 000 dernières images du train (le test n'est pas utilisé pour choisir lr)
    X, y = train_data
    tr, va = (X[:50000], y[:50000]), (X[50000:], y[50000:])
    acc = {n: [] for n in METHODES}
    for nom in METHODES:
        for lr in GRILLE_LR:
            a = np.mean([run(nom, lr, tr, va, EPOCHS_LR, s)["val_acc"][-1]
                         for s in range(N_SEEDS_LR)])
            acc[nom].append(a)
            print(f"{nom:8s} lr={lr:<8g} -> précision validation = {a:.4f}")
 
    best_lr = {n: GRILLE_LR[int(np.argmax(acc[n]))] for n in METHODES}
    print("Meilleur lr :", best_lr)
 
    plt.figure(figsize=(7, 4.5))
    for nom in METHODES:
        plt.plot(GRILLE_LR, acc[nom], marker="o", label=nom)
    plt.xscale("log")
    plt.xlabel("Learning rate")
    plt.ylabel(f"Précision validation (après {EPOCHS_LR} époques)")
    plt.title("Sensibilité au learning rate")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("exp1_sensibilite_lr.png", dpi=150)
    plt.close()
    return best_lr



**Expérience 2 : Courbes de convergence**  
Démontre la vitesse de convergence vers un minimum de F.

On prend en paramètre le meilleur Learning Rate pour chaque méthodes (calculé à l'expérience précédente).    
Perte d'entraînement : vitesse de convergence  
Précision test : qualité finale  
Temps cumulé : coût de calcul  
Taille moyenne des pas  
N_SEEDS fixé à 3 : compromis entre fiabilité  (bandes d'erreurs plus fiables) et temps de calucl

In [ ]:

def experiment_convergence(train_data, test_data, best_lr):
    results = {}
    # Lance 3 (N_SEEDS) entraînements complets pour chaque méthode. Chacun a un seed différent. Sur 10 epochs
    for nom in METHODES:
        runs = [run(nom, best_lr[nom], train_data, test_data, EPOCHS, s) for s in range(N_SEEDS)]
        results[nom] = {k: np.array([r[k] for r in runs]) for k in runs[0]}
        r = results[nom]
        print(f"{nom:8s} lr={best_lr[nom]:<6g} | précision test finale = "
              f"{r['val_acc'][:, -1].mean():.4f} +/- {r['val_acc'][:, -1].std():.4f} | "
              f"temps total = {r['time'][:, -1].mean():.1f}s")
 
    epochs = np.arange(1, EPOCHS + 1)
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    panneaux = [("train_loss", "Perte d'entraînement (échelle log)"),
                ("val_acc", "Précision test"),
                ("time", "Temps cumulé (s)"),
                ("deplacement", "Taille moyenne des pas  ||w_{k+1} - w_k||")]
    for ax, (key, titre) in zip(axes.ravel(), panneaux):
        for nom in METHODES:
            m, s = results[nom][key].mean(0), results[nom][key].std(0)
            ax.plot(epochs, m, marker="o", label=f"{nom} (lr={best_lr[nom]:g})")
            ax.fill_between(epochs, m - s, m + s, alpha=0.2)
        ax.set_title(titre)
        ax.set_xlabel("Époque")
        ax.grid(alpha=0.3)
    axes[0, 0].set_yscale("log")
    axes[1, 1].set_yscale("log")
    axes[0, 0].legend()
    plt.tight_layout()
    plt.savefig("exp2_convergence.png", dpi=150)
    plt.close()


Partie explication sur les différences entre les méthodes quant à :  
- La vitesse de convergence (nbr d'itérations / epochs)  
- La stabilité de l'entraînement (oscillation / divergence)  
- Qualité du minimum atteint  
- Sensibilité aux hyperparamètres (notamment Learning Rate)

**Partie optionnelle**  
  
Comparaison entre règles de MAJ codées à la main en numpy (comme dans TD d'opti) et résultats avec torch.optim
 

In [ ]:
def experiment_verification(train_data, test_data, best_lr):
    print("\nVérification : règles numpy vs torch.optim")
    plt.figure(figsize=(7, 4.5))
    for nom in METHODES:
        h_np = run(nom, best_lr[nom], train_data, test_data, EPOCHS_VERIF, seed=0, backend="numpy")
        h_th = run(nom, best_lr[nom], train_data, test_data, EPOCHS_VERIF, seed=0, backend="torch")
        ecart = np.max(np.abs(np.array(h_np["train_loss"]) - np.array(h_th["train_loss"])))
        print(f"{nom:8s} | écart max sur la perte d'entraînement = {ecart:.2e} | "
              f"précision test : numpy {h_np['val_acc'][-1]:.4f} vs torch {h_th['val_acc'][-1]:.4f}")
        ep = np.arange(1, EPOCHS_VERIF + 1)
        l, = plt.plot(ep, h_np["train_loss"], marker="o", label=f"{nom} (numpy)")
        plt.plot(ep, h_th["train_loss"], "k--", linewidth=1)
    plt.plot([], [], "k--", linewidth=1, label="torch.optim")
    plt.yscale("log")
    plt.xlabel("Époque")
    plt.ylabel("Perte d'entraînement")
    plt.title("Nos règles (numpy) vs torch.optim")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("exp3_verification.png", dpi=150)
    plt.close()



**Appel des fonctions**

In [ ]:
if __name__ == "__main__":
    train_data, test_data = load_mnist()
    best_lr = experiment_lr_sensitivity(train_data)
    experiment_convergence(train_data, test_data, best_lr)
    experiment_verification(train_data, test_data, best_lr)
    print("Figures : exp1_sensibilite_lr.png, exp2_convergence.png, exp3_verification.png")
